# Project Bài 6 · Tự viết cây quyết định dự báo đơn trễ

**Học máy cơ bản · Bài 6 · PGS.TS Nguyễn Văn Hậu**

**Tình huống.** Như Bài 5: lúc duyệt đơn, bộ phận chăm sóc khách hàng muốn biết đơn Olist nào có nguy cơ trễ. Lần này bạn tự viết thuật toán cây quyết định CART: Gini, quét ngưỡng bằng tổng tích luỹ, dựng cây đệ quy, dự đoán bằng cách lần đường đi. Rồi chọn độ sâu và ngưỡng đúng quy trình, chấm test một lần, so với logistic Bài 5.

**Dữ liệu.** `data/olist_model_table.csv`: cùng bảng, cùng sáu đặc trưng, cùng ba tập với Bài 4–5 (train 29.031 · validation 10.389 · test 40.266 đơn).

| Mốc | Bước | Ô tự kiểm |
|---|---|---|
| M1 | 1–2: dữ liệu; `gini`, `diem_chia` | số đơn; Gini của bộ 12 đơn (4/9, 2/9, 10/63) |
| M2 | 3: `quet_mot_cot` | 12 đơn: ≤ 650 km; gốc Olist: ≤ 1395.27 km, 25.817 đơn bên trái |
| M3 | 4 (khá): `dung_cay`, `du_doan` | trùng scikit-learn từng nút ở sâu 6 (45 nút); dự đoán validation trùng |
| M4 | 5: chọn độ sâu và τ trên validation; chấm test một lần | sâu 6; τ = 0,10; `cham_test` gọi đúng một lần |

**Ba mức.** *Đạt (5 điểm)*: bước 1–3, bước 5 (được dùng `DecisionTreeClassifier`, đặt `LAM_MUC_KHA = False`) và báo cáo một trang. *Khá (+3)*: bước 4, dùng cây tự viết ở bước 5. *Nâng cao (+2)*: một trong hai phần cuối (phân tích theo vùng so với logistic, hoặc tự viết cắt tỉa theo α).

**Luật chơi.** Ở mức khá, không dùng scikit-learn để dựng cây của bài nộp; chỉ dùng để đối chiếu ở ô có ghi rõ. **Chọn độ sâu và τ chỉ trên validation.** **Test chỉ chấm một lần** bằng `cham_test`. Chi phí FP + 5·FN là giả định lớp học.

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
from fractions import Fraction
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

ROOT = Path.cwd()
assert (ROOT / 'data' / 'olist_model_table.csv').exists(), 'Mở notebook từ thư mục bài (có thư mục data/).'
FEAT = ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g']

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')
LAM_MUC_KHA = True

## Bước 1 · Dữ liệu và ba tập

Median để điền thiếu chỉ học trên train. Cây không cần chuẩn hoá; ta vẫn chuẩn hoá (fit trên train) và ép kiểu float32 như scikit-learn, để ở bước 4 có thể so từng nút với scikit-learn.

In [ ]:
d = pd.read_csv(ROOT / 'data' / 'olist_model_table.csv')
tr, va, te = (d[d.split == k].reset_index(drop=True) for k in ['train', 'validation', 'test'])
# TODO: điền thiếu bằng median HỌC TRÊN TRAIN, rồi tạo RAW = {'train': ..., 'validation': ..., 'test': ...} (mảng numpy 6 cột)
raise NotImplementedError('TODO bước 1: tạo RAW rồi xoá dòng raise này')
sc = StandardScaler().fit(RAW['train'])
X = {k: sc.transform(v).astype(np.float32).astype(np.float64) for k, v in RAW.items()}
y = {'train': tr.late.to_numpy(), 'validation': va.late.to_numpy(), 'test': te.late.to_numpy()}

In [ ]:
# TỰ KIỂM · bước 1
kiem('Ba tập 29.031 / 10.389 / 40.266 đơn', (len(tr), len(va), len(te)) == (29031, 10389, 40266), 'Không lọc thêm hàng nào.')
kiem('Train có 941 đơn trễ', int(y['train'].sum()) == 941, '')
kiem('Không còn giá trị thiếu sau khi điền', not np.isnan(RAW['train']).any() and not np.isnan(RAW['test']).any(), 'Dùng median của train cho cả ba tập.')
kiem('Median học trên train', np.allclose(imp.statistics_, tr[FEAT].median()), 'fit SimpleImputer trên tr[FEAT].')

## Bước 2 · Gini và Gini sau chia (Chặng 2)

- `gini(n, k)`: Gini của một nút có n đơn, k đơn trễ: 1 − p² − (1 − p)².
- `diem_chia(nl, kl, nr, kr)`: Gini sau chia **có trọng số**: (nl/n)·G_L + (nr/n)·G_R.

Viết sao cho chạy được với số nguyên, `Fraction` và mảng numpy.

In [ ]:
def gini(n, k):
    # TODO: Gini nhị phân từ số đếm
    raise NotImplementedError('TODO bước 2: gini')

def diem_chia(nl, kl, nr, kr):
    # TODO: trung bình CÓ TRỌNG SỐ của Gini hai con
    raise NotImplementedError('TODO bước 2: diem_chia')

In [ ]:
# TỰ KIỂM · bước 2 (bộ 12 đơn mô phỏng ở slide; dùng Fraction để so chính xác)
F = Fraction
kiem('G gốc 12 đơn (4 trễ) = 4/9', gini(F(12), F(4)) == F(4, 9), '1 − (1/3)² − (2/3)²')
kiem('≤ 650 km: Δ = 2/9', gini(F(12), F(4)) - diem_chia(F(6), F(0), F(6), F(4)) == F(2, 9), 'trái 6 đơn 0 trễ; phải 6 đơn 4 trễ')
kiem('hứa ≤ 15: Δ = 10/63', gini(F(12), F(4)) - diem_chia(F(7), F(4), F(5), F(0)) == F(10, 63), '')
kiem('Bẫy ≤ 1.150 km: có trọng số = 4/11', diem_chia(F(11), F(3), F(1), F(1)) == F(4, 11), 'Bạn có quên trọng số không?')
kiem('Nút 9 đúng hạn, 3 trễ: G = 0,375', abs(gini(12, 3) - 0.375) < 1e-12, '')

## Bước 3 · Quét một cột (Chặng 3)

`quet_mot_cot(x, y, m)`: sắp đơn theo `x`, đi một lượt, cộng dồn số đơn và số trễ bên trái; chỉ xét vị trí mà giá trị **đổi** (x[i+1] > x[i] + 1e-7) và hai con đều có ít nhất `m` đơn; ngưỡng là trung điểm. Trả về `(nguong, G_sau, n_trai)` của ứng viên có G_sau nhỏ nhất, hoặc `None` nếu không có ứng viên.

Không dùng vòng lặp Python trên từng ngưỡng khi n lớn: dùng `np.cumsum`.

In [ ]:
def quet_mot_cot(x, y, m=1):
    # TODO: sắp xếp, cộng dồn, lọc vị trí hợp lệ, tính diem_chia, lấy nhỏ nhất
    raise NotImplementedError('TODO bước 3: quet_mot_cot')

In [ ]:
# TỰ KIỂM · bước 3
km = np.array([100. * i for i in range(1, 13)]); ngay = np.array([10, 20, 10, 20, 10, 20, 10, 10, 10, 20, 20, 10], float); tre = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1])
r = quet_mot_cot(km, tre)
kiem('12 đơn: cột khoảng cách chọn ≤ 650 km', r is not None and r[0] == 650 and r[2] == 6, 'Trung điểm giữa 600 và 700.')
kiem('12 đơn: G_sau = 2/9', abs(r[1] - 2 / 9) < 1e-12, '')
r = quet_mot_cot(RAW['train'][:, 0], y['train'], m=200)
kiem('Gốc Olist, khoảng cách: ngưỡng ≈ 1395.27 km', r is not None and abs(r[0] - 1395.2736142722893) < 1e-3, 'Nhớ ràng buộc mỗi con ≥ 200 đơn.')
kiem('Gốc Olist: 25.817 đơn bên trái', r[2] == 25817, '')
tot = [quet_mot_cot(RAW['train'][:, j], y['train'], m=200) for j in range(6)]
kiem('Gốc Olist: khoảng cách thắng cả sáu cột', int(np.argmin([t[1] for t in tot])) == 0, '')

## Bước 4 (mức khá) · Dựng cây đệ quy và dự đoán

`dung_cay(X, y, max_depth, m)` trả về **danh sách nút theo tiền thứ tự** (gốc, cả cây con trái, rồi cả cây con phải), mỗi nút là dict có `n`, `k` (số trễ), `p`, `feature` (−2 nếu là lá), `threshold`, `left`, `right` (−1 nếu là lá).

Dừng khi: đủ sâu, nút thuần, n < 2m, hoặc không có ứng viên ở mọi cột. Ở mỗi nút chọn cột có G_sau nhỏ nhất (hoà: giữ cột có chỉ số nhỏ hơn). `du_doan(cay, X)` lần đường đi, trả về p của lá.

In [ ]:
def dung_cay(X, y, max_depth, m, do_sau=0, cay=None):
    # TODO: thêm nút hiện tại; kiểm điều kiện dừng; quét mọi cột; chia; gọi lại trái rồi phải
    raise NotImplementedError('TODO bước 4: dung_cay')

def du_doan(cay, X):
    # TODO: lần đường đi từ gốc tới lá cho từng hàng
    raise NotImplementedError('TODO bước 4: du_doan')

In [ ]:
# TỰ KIỂM · bước 4 (chỉ chạy ở mức khá). scikit-learn chỉ dùng để ĐỐI CHIẾU.
if LAM_MUC_KHA:
    from sklearn.tree import DecisionTreeClassifier
    cay6 = dung_cay(X['train'], y['train'], 6, 200)
    sk = DecisionTreeClassifier(max_depth=6, min_samples_leaf=200, random_state=42).fit(X['train'], y['train']).tree_
    kiem('Cây tự viết có 45 nút như scikit-learn', len(cay6) == sk.node_count == 45, 'Thứ tự nút phải là tiền thứ tự.')
    kiem('Trùng từng nút: số đơn và cột', all(a['n'] == sk.n_node_samples[i] and a['feature'] == sk.feature[i] for i, a in enumerate(cay6)), 'Kiểm ràng buộc lá và quy tắc trung điểm.')
    kiem('Trùng ngưỡng (sai khác < 1e-6)', all(a['feature'] < 0 or abs(a['threshold'] - sk.threshold[i]) < 1e-6 for i, a in enumerate(cay6)), '')
    p_sk = DecisionTreeClassifier(max_depth=6, min_samples_leaf=200, random_state=42).fit(X['train'], y['train']).predict_proba(X['validation'])[:, 1]
    kiem('Dự đoán validation trùng scikit-learn', np.allclose(du_doan(cay6, X['validation']), p_sk, atol=1e-12), '')
else:
    print('Mức đạt: bỏ qua bước 4.')

## Bước 5 · Chọn trên validation, chấm test một lần (Chặng 4)

1. Lưới độ sâu {3; 6}, mỗi lá ≥ 200 đơn; chọn theo **AP validation** (hoà: ít lá hơn).
2. Khoá cây; quét τ ∈ {0,05; 0,10; 0,15; 0,20; 0,30; 0,50} theo chi phí giả định FP + 5·FN trên validation (hoà: ít cảnh báo hơn, rồi τ lớn hơn).
3. Chấm test **một lần**. Test này đã được xem ở bản thử nghiệm, v2 và Bài 5 (test lịch sử).

In [ ]:
def mo_hinh(do_sau):
    # Mức khá: cây tự viết; mức đạt: được dùng scikit-learn
    if LAM_MUC_KHA:
        cay = dung_cay(X['train'], y['train'], do_sau, 200)
        return (lambda Z: du_doan(cay, Z)), sum(a['left'] < 0 for a in cay)
    from sklearn.tree import DecisionTreeClassifier
    q = DecisionTreeClassifier(max_depth=do_sau, min_samples_leaf=200, random_state=42).fit(X['train'], y['train'])
    return (lambda Z: q.predict_proba(Z)[:, 1]), q.get_n_leaves()

def ma_tran(yy, canh_bao):
    return int(((~canh_bao) & (yy == 0)).sum()), int((canh_bao & (yy == 0)).sum()), int(((~canh_bao) & (yy == 1)).sum()), int((canh_bao & (yy == 1)).sum())

LUOI_SAU, LUOI_TAU = [3, 6], [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
# TODO: chọn DO_SAU theo AP validation (hoà: ít lá hơn) và hàm du_bao tương ứng; rồi chọn TAU theo FP + 5·FN trên validation
raise NotImplementedError('TODO bước 5: chọn DO_SAU, du_bao, TAU')
print('độ sâu khoá:', DO_SAU, '· τ khoá:', TAU)

In [ ]:
# TỰ KIỂM · bước 5a
kiem('Validation chọn sâu 6', DO_SAU == 6, 'AP validation của sâu 6 cao hơn sâu 3.')
kiem('Validation chọn τ = 0,10', TAU == 0.10, 'Quy tắc hoà: ít cảnh báo hơn rồi τ lớn hơn.')

In [ ]:
_SO_LAN_CHAM = 0
def cham_test(tau):
    # Chấm test một lần cho cây và τ ĐÃ KHOÁ trên validation
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã chấm một lần. Muốn thử thêm: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    return ma_tran(y['test'], du_bao(X['test']) >= tau)

KQ = cham_test(TAU)
tn, fp, fn, tp = KQ
print('Test · TN, FP, FN, TP =', KQ, '· precision', round(tp / (tp + fp), 4), '· recall', round(tp / (tp + fn), 4), '· chi phí', fp + 5 * fn, 'so với không cảnh báo', 5 * y['test'].sum())

In [ ]:
# TỰ KIỂM · bước 5b
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Restart & Run All nếu đã gọi nhiều lần.')
kiem('Test: (TN, FP, FN, TP) = (34659, 2129, 3046, 432)', KQ == (34659, 2129, 3046, 432), '')
kiem('Chi phí test thấp hơn không cảnh báo (17.390)', fp + 5 * fn < 17390, '')

## Báo cáo một trang (nộp cùng notebook)

Mỗi mục 2–4 câu.

1. **Dữ liệu.** Ba tập, số đơn, tỷ lệ trễ; nhãn theo ngày lịch; thời điểm dự đoán.
2. **Thuật toán.** Gini, quét trung điểm bằng tổng tích luỹ, điều kiện dừng; bằng chứng trùng scikit-learn (mức khá).
3. **Chọn trên validation.** Bảng AP theo độ sâu; bảng chi phí theo τ; quy tắc hoà.
4. **Test một lần.** Bốn ô; chi phí so với không cảnh báo; so với logistic Bài 5 (chi phí 16.944, AP 0.159).
5. **Giới hạn.** Ít nhất ba: test lịch sử; tỷ lệ trễ đổi theo thời gian (p ở lá là tần suất train); chi phí giả định; lưới độ sâu hẹp; không nhân quả.

## Nâng cao (chọn một)

- **A · Theo vùng.** Dùng hai tầng đầu của cây (gần/xa ở khoảng 1395 km; hứa ngắn ở nhánh gần và nhánh xa) để chia validation và test thành bốn vùng. So tỷ lệ trễ thật, p trung bình của cây và của logistic Bài 5 (`data/olist_*_tree_scores_v3.csv`, cột `p_logit`). So **mức chênh** giữa vùng xa · hứa ngắn và xa · hứa dài theo thực tế và theo từng mô hình. Vùng nào đứng vững qua thời gian?
- **B · Cắt tỉa theo α.** Với cây sâu 6 tự viết, tính α hiệu dụng (R(t) − R(T_t))/(số lá của T_t − 1) cho mọi nút trong; cắt nhánh yếu nhất lặp lại; so dãy α với `cost_complexity_pruning_path` của scikit-learn.

In [ ]:
# Nâng cao A · khởi đầu (tuỳ chọn)
diem = pd.read_csv(ROOT / 'data' / 'olist_validation_tree_scores_v3.csv')
print(diem.groupby('region').agg(so_don=('late', 'size'), ty_le_tre=('late', 'mean'), p_cay=('p_tree', 'mean'), p_logistic=('p_logit', 'mean')).round(4))